# 07 · NGSIM stop-and-go — Spacing

Two-shot decomposition of **spacing** (leader position − follower position) for
the NGSIM stop-and-go trajectories. Trajectories with any negative spacing are
discarded. As with speed, the reconstruction is clipped at zero and a strictly
positive personalized component is derived. Clustering here is performed on the
*reconstructed* spacing (top-5 PCA scores, k-means with k = 3).

**Figures produced:** `NGSIM_space_component.png`, `NGSIM_space_compare.png`,
`NGSIM_space_cluster.png`, `NGSIM_cluster_space.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from bhd import DATA_DIR, FIG_DIR, set_style, load_ngsim_stop_to_go

set_style(seaborn_white=True)

## 1. Load, filter, and align trajectories

In [ ]:
df = load_ngsim_stop_to_go(DATA_DIR / "ngsim" / "stop_to_go.csv")

valid_traj_ids = [tid for tid in df["trajectory_id"].unique() if (df.loc[df["trajectory_id"] == tid, "spacing"] >= 0).all()]
df_valid = df[df["trajectory_id"].isin(valid_traj_ids)]
min_len = min(len(df_valid[df_valid["trajectory_id"] == tid]) for tid in valid_traj_ids)
print(f"{len(valid_traj_ids)} valid trajectories (non-negative spacing), resampled to {min_len} points")

spacing_list = []
for tid in valid_traj_ids:
    spacing_vals = df_valid[df_valid["trajectory_id"] == tid]["spacing"].values
    f_spacing = interp1d(np.linspace(0, 1, len(spacing_vals)), spacing_vals)
    spacing_list.append(f_spacing(np.linspace(0, 1, min_len)))
S = np.stack(spacing_list, axis=1)  # (time, vehicles)

## 2. Two-shot decomposition with positivity constraint

In [ ]:
pca_shared = PCA(n_components=1)
C_spacing = pca_shared.fit(S.T).components_.T
Psi_spacing = C_spacing.T @ S
S_shared = C_spacing @ Psi_spacing

personalized_spacing, reconstructed_spacing = [], []
for v in range(S.shape[1]):
    R_v = (S[:, v] - S_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    R_hat = pca_personal.inverse_transform(Phi_v)
    personalized_spacing.append(R_hat.flatten())
    reconstructed_spacing.append(S_shared[:, v] + R_hat.flatten())
S_personal = np.column_stack(personalized_spacing)
S_reconstructed = np.column_stack(reconstructed_spacing)

S_reconstructed_clipped = np.clip(S_reconstructed, a_min=0, a_max=None)
S_personal_corrected = np.clip(S_reconstructed_clipped - S_shared, a_min=0, a_max=None)

colors = cm.viridis(np.linspace(0, 1, S.shape[1]))
xticks = list(range(0, 401, 50))
xtick_labels = [str(x // 10) for x in xticks]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for i in range(S.shape[1]):
    axes[0].plot(S[:, i], color=colors[i], alpha=0.4)
    axes[1].plot(S_shared[:, i], color=colors[i], alpha=0.6)
    axes[2].plot(S_personal_corrected[:, i], linestyle="--", color=colors[i], alpha=0.8)
axes[0].set_title("True Spacing", fontsize=22)
axes[1].set_title("Common Component (CΨ)", fontsize=22)
axes[2].set_title("Strictly Positive Personalized Component (PΦ)", fontsize=22)
axes[2].set_xlabel("Time (sec)", fontsize=22)
for ax in axes:
    ax.tick_params(axis="both", labelsize=20)
    ax.set_ylim(0, 60)
    ax.set_xlim(0, 400)
    ax.set_yticks([0, 20, 40, 60])
    ax.set_ylabel("Spacing (m)", fontsize=20)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels)
    ax.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_space_component.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. True vs. reconstructed spacing

In [ ]:
plt.figure(figsize=(10, 7))
for i in range(S.shape[1]):
    plt.plot(S[:, i], color=colors[i], alpha=0.25, linewidth=4)
    plt.plot(S_reconstructed[:, i], linestyle="--", color=colors[i], linewidth=2)
plt.plot([], [], color="gray", linewidth=7, alpha=0.25, label="True")
plt.plot([], [], linestyle="--", color="gray", linewidth=4, label="Reconstructed")
plt.xlabel("Time (sec)", fontsize=22)
plt.xlim(0, 400)
plt.xticks(ticks=xticks, labels=xtick_labels, fontsize=20)
plt.ylabel("Spacing (m)", fontsize=22)
plt.yticks([10, 20, 30, 40, 50], fontsize=20)
plt.ylim(-2, 56)
plt.legend(fontsize=22, loc="lower center", ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_space_compare.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Clustering the reconstructed spacing

In [ ]:
pca_cluster = PCA(n_components=5)
X_cluster = pca_cluster.fit_transform(S_reconstructed.T)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_cluster)
print("Cluster sizes:", np.bincount(labels))

plt.figure(figsize=(10, 7))
palette = sns.color_palette("Set2", n_colors=3)
for cid in range(3):
    idx = np.where(labels == cid)[0]
    plt.plot(S_reconstructed[:, idx].mean(axis=1), label=f"Cluster {cid}", color=palette[cid], linewidth=3)
plt.xlabel("Time (sec)", fontsize=22)
plt.xticks(ticks=xticks, labels=xtick_labels, fontsize=20)
plt.xlim(0, 400)
plt.ylabel("Spacing (m)", fontsize=22)
plt.yticks(fontsize=20)
plt.ylim(5, 56)
plt.legend(fontsize=22, loc="lower center", ncol=3)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_space_cluster.png", dpi=300, bbox_inches="tight")
plt.show()

spacing_2d = PCA(n_components=2).fit_transform(S_reconstructed.T)
plt.figure(figsize=(10, 10))
sns.scatterplot(x=spacing_2d[:, 0], y=spacing_2d[:, 1], hue=labels, palette="Set2", s=120, edgecolor="black")
for i in range(len(labels)):
    plt.text(spacing_2d[i, 0] + 0.03, spacing_2d[i, 1], f"V{i + 1}", fontsize=20)
handles, legend_labels = plt.gca().get_legend_handles_labels()
new_labels = [f"Cluster {int(lbl)}" if lbl.isdigit() else lbl for lbl in legend_labels]
plt.xlabel("Personalized Component 1", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Personalized Component 2", fontsize=22)
plt.yticks(fontsize=20)
plt.legend(handles, new_labels, fontsize=22)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_cluster_space.png", dpi=300, bbox_inches="tight")
plt.show()